# Import data

## Import player position/rotation data
Read from .json and build dataframe

In [4]:
import json
import pandas as pd
import datetime as dt

FILENAME: str = "../data/savefile_01_pos_rot.json"
 
# 1. Load the raw JSON
with open(FILENAME, mode="r") as file:
    json_data: json = json.load(file)

# 2. Extract top-level metadata
player_id: str = json_data["playerId"]
save_time: dt.datetime = dt.datetime.strptime(json_data["saveDateTime"], "%Y-%m-%d %H:%M:%S")

# 3. Build rows from the list, reshaping position/rotation into tuples
rows: list[dict] = []
for entry in json_data["playerSaveObjectList"]:
    rows.append({
        "time":      entry["gametimer"],
        "position":  (entry["position"]["x"], entry["position"]["y"], entry["position"]["z"]),
        "rotation":  (entry["rotation"]["x"], entry["rotation"]["y"], entry["rotation"]["z"]),
        "playerId":  player_id,
        "saveTime":  save_time,
    })

# 4. Create the DataFrame
df: pd.DataFrame = pd.DataFrame(rows)

x_list: list[float] =  [x for (x,y,z) in df["position"]]
z_list: list[float] =  [z for (x,y,z) in df["position"]]
rot_y_list: list[float] = [y for (x,y,z) in df["rotation"]]
t_list: list[float] = [_ for _ in df["time"]]

print(df.head())
print("-----------")
print(df.dtypes)

       time           position                        rotation playerId  \
0  5.528239  (90.0, 0.0, 60.0)   (0.0, 46.83423614501953, 0.0)   id_001   
1  6.530888  (90.0, 0.0, 60.0)  (0.0, 107.00001525878906, 0.0)   id_001   
2  7.531301  (90.0, 0.0, 60.0)   (0.0, 95.61361694335938, 0.0)   id_001   
3  8.531610  (90.0, 0.0, 60.0)   (0.0, 95.61361694335938, 0.0)   id_001   
4  9.531744  (90.0, 0.0, 60.0)   (0.0, 87.09896087646484, 0.0)   id_001   

             saveTime  
0 2026-07-26 20:11:07  
1 2026-07-26 20:11:07  
2 2026-07-26 20:11:07  
3 2026-07-26 20:11:07  
4 2026-07-26 20:11:07  
-----------
time               float64
position            object
rotation            object
playerId               str
saveTime    datetime64[us]
dtype: object


## Import building data
Read from .json and import as Dataframe

In [5]:
BUILDING_FILENAME: str = "../data/buildings.json"
buildings_data: pd.DataFrame = pd.read_json(BUILDING_FILENAME).set_index("building_id")
print(buildings_data)

                   name door_orientation validation_color validation_symbol  \
building_id                                                                   
1               Butcher            South              red            square   
2                  Barn             East           orange            circle   
3                Potter             East            white          triangle   
4              Windmill            North            green              star   
5            Blacksmith             East            brown              moon   
6                Tavern            South             pink             cross   
7                Bakery             West           purple             arrow   
8                Tailor            North           yellow        semicircle   
9                Cooper             West             blue             cloud   

             grid_pos_x  grid_pos_z  world_pos_x  world_pos_z  
building_id                                                    
1 

## Import attemps data


In [118]:
import json
ATTEMPTS_FILENAME: str = "../data/savefile_02_attempts.json"

with open(ATTEMPTS_FILENAME, mode="r") as file:
    json_data: json = json.load(file)

trial_rows: list[dict] = []
for entry in json_data["trialSaveObjectList"]:
    trial_rows.append({
         "time": float(entry["gameTimer"]),
         "attempts_count": entry["currentPhase"]["attemptsCount"],
         "target_building_id": entry["currentPhase"]["targetBuildingId"]
    })

trial_df: pd.DataFrame = pd.DataFrame(trial_rows)
print(trial_df)


def get_attempt_num_for_time(time: float) -> int:
    """
    Returns the attempt number for a given play time.
    If there is no attempt number for the play time, returns '-1'
    """
    try:
        lowest_time_value_index: int = trial_df.loc[(trial_df.time < time)]["time"].idxmax()
        print("idmax", lowest_time_value_index)
    except ValueError:
        return -1
    
    return int(trial_df.loc[lowest_time_value_index]["attempts_count"]) #  without casting, returns np.int64(2)

def get_target_building_id_for_time(time: float) -> str:
    """
    Returns the attempt number for a given play time.
    If there is no attempt number for the play time, returns '-1'
    """
    try:
        lowest_time_value_index: int = trial_df.loc[(trial_df.time < time)]["time"].idxmax()

    except ValueError:
        return -1
    
    return trial_df.loc[lowest_time_value_index]["target_building_id"] #  casting not required for strings



# # 3. Build rows from the list, reshaping position/rotation into tuples
# rows: list[dict] = []
# for entry in json_data["playerSaveObjectList"]:
#     rows.append({
#         "time":      entry["gameTimer"],
#         "position":  (entry["position"]["x"], entry["position"]["y"], entry["position"]["z"]),
#         "rotation":  (entry["rotation"]["x"], entry["rotation"]["y"], entry["rotation"]["z"]),
#         "playerId":  player_id,
#         "saveTime":  save_time,
#     })

# # 4. Create the DataFrame
# df: pd.DataFrame = pd.DataFrame(rows)

# x_list: list[float] =  [x for (x,y,z) in df["position"]]
# z_list: list[float] =  [z for (x,y,z) in df["position"]]
# rot_y_list: list[float] = [y for (x,y,z) in df["rotation"]]
# t_list: list[float] = [_ for _ in df["time"]]

get_attempt_num_for_time(30)
get_target_building_id_for_time(30)


        time  attempts_count target_building_id
0  28.460131               2              L1-01
1  43.185093               3              L1-01
2  77.502213               4              L1-01
idmax 0


'L1-01'

# Model with charts

## Map configuration for Level 1

In [4]:
# MAP SIZE
GRID_COLS: int = 6
GRID_ROWS: int = 5
BLOCK_SIZE: int = 30

# The extent is computed — maps pixel corners to world coordinates
map_extent = [
    -BLOCK_SIZE / 2,
    (GRID_COLS - 0) * BLOCK_SIZE + BLOCK_SIZE / 2,
    -BLOCK_SIZE / 2,
    (GRID_ROWS - 0) * BLOCK_SIZE + BLOCK_SIZE / 2,
]

map_x_offset: int = BLOCK_SIZE / 2
map_z_offset: int = BLOCK_SIZE / 2

map_coord: list[int] = [
    -BLOCK_SIZE / 2,
    (GRID_COLS - 1) * BLOCK_SIZE + BLOCK_SIZE / 2,
    -BLOCK_SIZE / 2,
    (GRID_ROWS - 1) * BLOCK_SIZE + BLOCK_SIZE / 2,
]

map_coord[0] += map_x_offset  # left
map_coord[1] += map_x_offset  # right
map_coord[2] += map_z_offset  # bottom
map_coord[3] += map_z_offset  # top

## Map Graph

### Map Graph (plotly)

In [ ]:
import plotly.graph_objects as go
from PIL import Image
import numpy as np

# Load map image
map_img = Image.open("../images/map_level1.png")

SIZE = 4
base_triangle = np.array([
    [0, SIZE/2],
    [-SIZE/2, -SIZE/2],
    [SIZE/2, -SIZE/2],
])

# --- Build the figure with initial traces ---

fig = go.Figure()

# Trace 0: trail (static — never changes)
fig.add_trace(go.Scatter(
    x=x_list, y=z_list,
    mode='markers',
    marker=dict(color='white', size=5, opacity=0.25),
    name='Trail',
    showlegend=False
))

# Trace 1: buildings with labels (static — toggle via legend click)
fig.add_trace(go.Scatter(
    x=buildings_data["world_pos_x"].tolist(),
    y=buildings_data["world_pos_z"].tolist(),
    mode='markers+text',
    marker=dict(color='orange', size=12, symbol='square',
                line=dict(color='black', width=1)),
    text=buildings_data["name"].tolist(),
    textposition='top right',
    textfont=dict(size=9, color='black'),
    name='POIs'
))

# Trace 2: path (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='lines+markers',
    marker=dict(color='cyan', size=4),
    line=dict(color='cyan', width=1),
    showlegend=False
))

# Trace 3: current position marker (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='markers',
    marker=dict(color='red', size=12),
    showlegend=False
))

# Trace 4: rotation triangle (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    fill='toself',
    fillcolor='red',
    line=dict(color='white', width=1.5),
    showlegend=False
))

# --- Build all animation frames upfront ---

frames = []
for i in range(len(x_list)):
    # Rotation triangle
    next_frame = min(i + 1, len(x_list) - 1)
    angle = -rot_y_list[i]
    rad = np.radians(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    rotated = np.array([
        [v[0]*cos_a - v[1]*sin_a, v[0]*sin_a + v[1]*cos_a]
        for v in base_triangle
    ])
    rotated[:, 0] += x_list[next_frame]
    rotated[:, 1] += z_list[next_frame]
    tri_x = list(rotated[:, 0]) + [rotated[0, 0]]  # close the shape
    tri_y = list(rotated[:, 1]) + [rotated[0, 1]]

    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_list[:i+1], y=z_list[:i+1]),      # path
            go.Scatter(x=[x_list[i]], y=[z_list[i]]),          # current
            go.Scatter(x=tri_x, y=tri_y),                     # triangle
        ],
        traces=[2, 3, 4],  # which trace indices to update
        name=str(i)
    ))

fig.frames = frames

# --- Background image ---

fig.add_layout_image(
    source=map_img,
    xref="x", yref="y",
    x=map_coord[0],
    y=map_coord[3],       # plotly anchors images from top-left
    sizex=map_coord[1] - map_coord[0],
    sizey=map_coord[3] - map_coord[2],
    sizing="stretch",
    layer="below"
)

# --- Layout: axes, play button, slider ---

fig.update_layout(
    width=800, height=700,
    title="Player Position over Time",
    xaxis=dict(range=[map_extent[0], map_extent[1]], dtick=30, showgrid=True),
    yaxis=dict(range=[map_extent[2], map_extent[3]], dtick=30,
               showgrid=True, scaleanchor="x"),

    # Play/pause buttons
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.5, y=-0.05, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=100, redraw=True),
                                  fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )],

    # Time slider
    sliders=[dict(
        active=0,
        x=0.05, len=0.9,
        currentvalue=dict(prefix="Time: "),
        steps=[
            dict(
                args=[[str(i)], dict(frame=dict(duration=0, redraw=True),
                                      mode="immediate")],
                method="animate",
                label=f"{t_list[i]:.1f}s"
            )
            for i in range(len(x_list))
        ]
    )]
)

fig.show()